# ML Baseline And Fusion Search

This notebook runs fast strawberry-only LOOCV experiments before training the deep models. It compares sequence length and fusion style using image summary features plus timestamp-aligned temperature/humidity.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

LAB_DIR = Path.cwd()
if LAB_DIR.name != 'strawberry':
    LAB_DIR = Path('notebooks/strawberry').resolve()
sys.path.insert(0, str(LAB_DIR))
import lab_utils as lab


## Run Sweep

This may take a few minutes because it reads strawberry images and runs LOOCV over the available fruit IDs.


In [ ]:
results = lab.run_ml_baseline_sweep()
results.head(20)


In [ ]:
results.sort_values(['fold_mae_mean', 'seq_len']).head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for fusion, group in results.groupby('fusion_mode'):
    ax.plot(group['seq_len'], group['fold_mae_mean'], marker='o', label=fusion)
ax.set_xlabel('seq_len')
ax.set_ylabel('LOOCV MAE (hours)')
ax.set_title('Sequence/fusion lab sweep')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()


In [ ]:
best = results.sort_values(['fold_mae_mean', 'seq_len']).iloc[0].to_dict()
best


## How To Interpret

Use LOOCV mean MAE and fold stability to choose the large-model config. The current sweep selected `seq_len=3` with `late_env_branch`, meaning sensor data is used as a side branch after visual temporal encoding rather than as raw early concatenation.
